In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

In [3]:
# Download latest version
sentiment140_path = ("/content/drive/MyDrive/Sentiment 140/sentiment_1600000_data.csv")

print("Path to dataset files:", sentiment140_path)

import kagglehub


imdb_path = ("/content/drive/MyDrive/IMDB/IMDB Dataset.csv")

print("Path to dataset files:", imdb_path)


Path to dataset files: /content/drive/MyDrive/Sentiment 140/sentiment_1600000_data.csv
Path to dataset files: /content/drive/MyDrive/IMDB/IMDB Dataset.csv


In [4]:
import os
import pandas as pd

# Define file paths inside the downloaded folders
sentiment140_file = os.path.join(sentiment140_path, "/content/drive/MyDrive/Sentiment 140/sentiment_1600000_data.csv")
imdb_file = os.path.join(imdb_path, "IMDB Dataset.csv")

# Read the CSV files
sentiment140_cols = ["target", "ids", "date", "flag", "user", "text"]
sentiment140_df = pd.read_csv(sentiment140_file, encoding='ISO-8859-1', names=sentiment140_cols)
imdb_df = pd.read_csv(imdb_path)


In [5]:
# Sample data
sentiment140_df = sentiment140_df[['text', 'target']]
sentiment140_df['target'] = sentiment140_df['target'].map({0: 'negative', 4: 'positive'})
sentiment140_sample = sentiment140_df.sample(n=100000, random_state=42)
imdb_sample = imdb_df.sample(n=25000, random_state=42).rename(columns={'review': 'text', 'sentiment': 'target'})

<ipython-input-5-2200173216>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentiment140_df['target'] = sentiment140_df['target'].map({0: 'negative', 4: 'positive'})


In [6]:
# Combine datasets
combined_df = pd.concat([sentiment140_sample, imdb_sample], ignore_index=True)

def clean_text(text):
    text = re.sub(r"@\S+|https?://\S+|[^A-Za-z0-9\s]+", ' ', str(text).lower())
    return text.strip()

combined_df['text'] = combined_df['text'].apply(clean_text)
label_encoder = LabelEncoder()
combined_df['target'] = label_encoder.fit_transform(combined_df['target'])

In [7]:
# Reset full index to prevent indexing issues
combined_df = combined_df.reset_index(drop=True)

# Split data
train_texts, test_texts, train_labels, test_labels = train_test_split(
    combined_df['text'], combined_df['target'], test_size=0.3, random_state=42
)

In [8]:
# Reset indices to avoid KeyError
train_texts = train_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
train_labels = train_labels.reset_index(drop=True)
test_labels = test_labels.reset_index(drop=True)

In [9]:
# Dataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [10]:
# Tokenizer and Dataloaders
MAX_LEN = 128
BATCH_SIZE = 64
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [11]:
# BERT Classifier
class BertClassifier(nn.Module):
    def __init__(self, n_classes):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-cased')
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        return self.out(self.drop(pooled_output))

In [25]:
# Training
EPOCHS = 10
model = BertClassifier(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [26]:
from torch.optim import AdamW

# Correct usage:
optimizer = AdamW(model.parameters(), lr=2e-5)


In [27]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss().to(device)


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [29]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss().to(device)


In [30]:
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    correct_train = 0

    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        total_train_loss += loss.item()

        with torch.no_grad():
            _, preds = torch.max(outputs, dim=1)
            correct_train += torch.sum(preds == labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    accuracy = (correct_train.double() / len(train_dataset)).item()
    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {total_train_loss:.4f}, Accuracy: {correct_train.double() / len(train_dataset):.4f}")


100%|██████████| 1368/1368 [06:19<00:00,  3.60it/s]


Epoch 1/10, Loss: 562.0935, Accuracy: 0.8108


100%|██████████| 1368/1368 [06:19<00:00,  3.60it/s]


Epoch 2/10, Loss: 413.5572, Accuracy: 0.8732


100%|██████████| 1368/1368 [06:19<00:00,  3.60it/s]


...

Epoch 8/10, Loss: 50.5916, Accuracy: 0.9880


100%|██████████| 1368/1368 [06:19<00:00,  3.60it/s]


Epoch 9/10, Loss: 42.8843, Accuracy: 0.9899


100%|██████████| 1368/1368 [06:19<00:00,  3.60it/s]

Epoch 10/10, Loss: 34.6733, Accuracy: 0.9924


In [32]:
from sklearn.metrics import confusion_matrix, classification_report

# Collect predictions and true labels
y_pred = []
y_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)

        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.cpu().numpy())

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


Confusion Matrix:
[[15668  3089]
 [ 3218 15525]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.84      0.83     18757
    positive       0.83      0.83      0.83     18743

    accuracy                           0.83     37500
   macro avg       0.83      0.83      0.83     37500
weighted avg       0.83      0.83      0.83     37500



In [33]:
# Save the model state dict
model_save_path = '/content/drive/MyDrive/Sentiment 140/sentiment_model_complete'
torch.save(model.state_dict(), model_save_path)

# Save the tokenizer
tokenizer_save_path = '/content/drive/MyDrive/Sentiment 140/tokenizer'
tokenizer.save_pretrained(tokenizer_save_path)

print(f"Model and tokenizer saved successfully to {model_save_path} and {tokenizer_save_path}")


Model and tokenizer saved successfully to /content/drive/MyDrive/Sentiment 140/sentiment_model_complete and /content/drive/MyDrive/Sentiment 140/tokenizer


In [36]:
# Load the model
model = BertClassifier(2)  # Make sure to use the same model class
model.load_state_dict(torch.load(model_save_path))
model.eval()  # Set the model to evaluation mode

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained(tokenizer_save_path)

print("Model and tokenizer loaded successfully.")


Model and tokenizer loaded successfully.


In [41]:
def evaluate_text(input_text, confidence_threshold=0.6):
    # Clean and tokenize the input text
    input_text = clean_text(input_text)
    encoding = tokenizer.encode_plus(
        input_text,
        add_special_tokens=True,
        max_length=128,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']

    # Ensure the model is in evaluation mode
    model.eval()

    # Get model output
    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = output[0]  # Access logits for binary classification

        # Apply softmax to get probabilities
        probabilities = torch.nn.functional.softmax(logits, dim=-1)

        # Print the logits and probabilities
        print(f"Logits: {logits}")
        print(f"Probabilities: {probabilities}")

        # Get the predicted class (index of max probability)
        predicted_class = torch.argmax(probabilities, dim=-1).item()

        # Check the model's confidence
        max_confidence = probabilities[predicted_class].item()  # Corrected indexing
        print(f"Predicted class: {predicted_class} with confidence: {max_confidence}")

        # Compare the confidence level with the threshold
        if max_confidence < confidence_threshold:
            print("Model is unsure about the sentiment.")
            return "Model is unsure about the sentiment."

    # Convert prediction to sentiment
    sentiment = 'positive' if predicted_class == 1 else 'negative'
    print(f"Sentiment: {sentiment}")

    return sentiment


In [44]:
# List of tough sentiment texts (both positive and negative)
sentiment_texts = [
    "The product is exceptional, but the delivery process was a nightmare. I had to wait for days without any updates.",
    "I absolutely love the customer service, but the product arrived damaged. Very frustrating!",
    "This is the best purchase I've made in a while. The quality exceeded my expectations and it's exactly what I needed.",
    "The experience was horrible. The product didn't match the description, and customer support was unresponsive.",
    "Highly recommend this product! It's everything I was looking for, and the quality is top-notch.",
    "I am so disappointed. The product broke after one use and the return process has been a hassle.",
    "Great value for money. I was skeptical at first, but this exceeded my expectations in every way.",
    "I regret buying this. It doesn't do what it promised and is a waste of money.",
    "Fantastic quality! Everything works perfectly and exceeded my expectations. Will definitely buy again.",
    "The product didn't live up to the hype. The quality is subpar, and it doesn't perform well. A complete letdown."
]

# Loop through the list of sentiment texts and evaluate each one
for input_text in sentiment_texts:
    sentiment = evaluate_text(input_text)  # Evaluate using the existing evaluate_text function
    print(f"Sentiment for text: {input_text}\nPredicted Sentiment: {sentiment}\n")


Logits: tensor([ 6.3596, -4.7779])
Probabilities: tensor([9.9999e-01, 1.4555e-05])
Predicted class: 0 with confidence: 0.9999854564666748
Sentiment: negative
Sentiment for text: The product is exceptional, but the delivery process was a nightmare. I had to wait for days without any updates.
Predicted Sentiment: negative

Logits: tensor([ 5.9055, -4.8121])
Probabilities: tensor([9.9998e-01, 2.2150e-05])
Predicted class: 0 with confidence: 0.9999778270721436
Sentiment: negative
Sentiment for text: I absolutely love the customer service, but the product arrived damaged. Very frustrating!
Predicted Sentiment: negative

Logits: tensor([-5.3809,  4.0372])
Probabilities: tensor([8.1231e-05, 9.9992e-01])
Predicted class: 1 with confidence: 0.9999188184738159
Sentiment: positive
Sentiment for text: This is the best purchase I've made in a while. The quality exceeded my expectations and it's exactly what I needed.
Predicted Sentiment: positive

Logits: tensor([ 6.2326, -4.7292])
Probabilities: t